# SafeCart Dataset Pipeline
### Evidence-gated genuine-reference vs. reported-counterfeit-candidate collector

This notebook builds `SafeCart_Dataset/` from the products in `product.xlsx`.

1. It collects genuine/reference images separately from official and trusted retail sources.
2. It examines publicly indexed marketplace listing/review evidence only through counterfeit-oriented queries.
3. It creates a counterfeit candidate only when explicit buyer-reported counterfeit evidence is traceable to a marketplace listing, and only downloads an image from that same listing when possible.
4. It never treats an ordinary marketplace listing, an unknown seller, a low price, or a low rating alone as counterfeit evidence.
5. It never automatically assigns `counterfeit_confirmed`; the automatic maximum is `reported_counterfeit_candidate`.

If a marketplace page or review cannot be accessed, the notebook records `review_access = unavailable`; it does not fabricate reviews. Run cells in order. Pilot mode processes five products by default.


## 1. Install dependencies

In [1]:
!pip install -q pillow imagehash pytesseract rapidfuzz pandas openpyxl tqdm requests beautifulsoup4
!apt-get -qq install -y tesseract-ocr > /dev/null
print("Dependencies installed.")


Dependencies installed.


The system cannot find the path specified.


## 2. Upload master dataset

Upload your `product.xlsx` (the file with columns: brand, product_name, product_type,
size, bpom_id, product_url, ingredients_list, description_product, etc.).

In [2]:
import pandas as pd
import re, os
from pathlib import Path

# Automatically select the first local XLSX that has the required master-product columns.
REQUIRED_COLS = ["brand", "product_name", "bpom_id"]
MASTER_XLSX_PATH = None
for candidate in sorted(Path(".").glob("*.xlsx")):
    try:
        preview = pd.read_excel(candidate, nrows=3)
        normalized = [str(c).strip().lower().replace(" ", "_") for c in preview.columns]
        if all(c in normalized for c in REQUIRED_COLS):
            MASTER_XLSX_PATH = str(candidate)
            break
    except Exception:
        continue
if not MASTER_XLSX_PATH:
    raise FileNotFoundError("No local XLSX containing brand, product_name, and bpom_id was found.")

master_df = pd.read_excel(MASTER_XLSX_PATH)
master_df.columns = [str(c).strip().lower().replace(" ", "_") for c in master_df.columns]
missing = [c for c in REQUIRED_COLS if c not in master_df.columns]
if missing: raise ValueError(f"Master dataset is missing: {missing}")

def slugify(text):
    text = re.sub(r"[^A-Z0-9]+", "_", str(text).strip().upper())
    return re.sub(r"_+", "_", text).strip("_")
def make_product_id(row): return f"{slugify(row['brand'])}_{slugify(row['product_name'])}_{slugify(row['bpom_id'])}"
master_df["product_id"] = master_df.apply(make_product_id, axis=1)
n_products = len(master_df)
print(f"Loaded {n_products} products from {MASTER_XLSX_PATH}.")
master_df[["product_id", "brand", "product_name", "bpom_id"]].head()


Loaded 91 products from product.xlsx.


,product_id,brand,product_name,bpom_id
0,COSRX_AHA_BHA_CLARIFYING_TREATMENT_TONER_NA262...,COSRX,AHA/BHA Clarifying treatment toner,NA26221200804
1,SUNGBOON_EDITOR_SUNGBOON_EDITOR_GREEN_TOMATO_P...,Sungboon Editor,Sungboon Editor Green Tomato Pore Lifting Ampo...,NA26251200757
2,DEAR_KLAIRS_FRESHLY_JUICED_VITAMIN_SKIN_PREP_P...,Dear Klairs,Freshly Juiced Vitamin Skin Prep Pads,NA26251200678
3,DEAR_KLAIRS_FRESHLY_JUICED_VITAMIN_ESSENCE_TON...,Dear Klairs,Freshly Juiced Vitamin Essence Toner,NA26251200677
4,I_M_FROM_RICE_TONER_PAD_NA26251200291,I'm From,Rice Toner Pad,NA26251200291


## 3. Configure Serper API

Paste the Serper API key directly in the next cell. It is sent only in request headers and is never written to CSVs, metadata, or cache files. **Do not share this notebook publicly after inserting the API key.**


In [ ]:
# Serper API configuration
# IMPORTANT: Do not share this notebook publicly after inserting the API key.
SERPER_API_KEY = ""
if not SERPER_API_KEY or SERPER_API_KEY == "PASTE_YOUR_SERPER_API_KEY_HERE":
    raise ValueError("Set SERPER_API_KEY in this cell before running the API test.")

BASE_DIR = "SafeCart_Dataset"
IMAGES_DIR, META_DIR, CACHE_DIR = os.path.join(BASE_DIR, "images"), os.path.join(BASE_DIR, "metadata"), os.path.join(BASE_DIR, "cache")
for d in [os.path.join(IMAGES_DIR, "genuine_reference"), os.path.join(IMAGES_DIR, "reported_counterfeit_candidate"),
          os.path.join(IMAGES_DIR, "unknown"), META_DIR, CACHE_DIR]: os.makedirs(d, exist_ok=True)

PILOT_MODE = False
PILOT_PRODUCTS = 5
GENUINE_TARGET = 8
COUNTERFEIT_MAX_CANDIDATES = 8
RESULTS_PER_QUERY = 10
MIN_WIDTH, MIN_HEIGHT = 300, 300
MAX_WORKERS = 3
REQUEST_DELAY_SEC, MAX_RETRIES, RETRY_BACKOFF_SEC, REQUEST_TIMEOUT_SEC = 0.3, 3, 1.5, 30
SERPER_API_TEST_PASSED = False
MAX_PUBLIC_EVIDENCE_PER_PRODUCT = 1  # pilot safeguard: retain diverse evidence without timing out on every source page

print(f"Serper configured; pilot mode: {PILOT_MODE} ({PILOT_PRODUCTS} products).")
print("Set PILOT_MODE = True after validation to process all products in the master XLSX.")


Serper configured; pilot mode: False (5 products).
Set PILOT_MODE = True after validation to process all products in the master XLSX.


In [4]:
# Configuration is in Cell 6. Run the Serper API test cell before the pipeline.


## 4. Serper search, caching, and error handling

Serper Google Search supplies public evidence snippets, and Serper Images supplies genuine images or images matched to an already-evidenced listing. Successful responses are cached under `SafeCart_Dataset/cache/` to prevent duplicate query charges.


In [5]:
import requests
import time, json, hashlib, unicodedata
from datetime import datetime, timezone
from urllib.parse import urlparse, urlunparse

SERPER_WEB_URL = "https://google.serper.dev/search"
SERPER_IMAGE_URL = "https://google.serper.dev/images"
MARKETPLACE_DOMAINS = ("shopee.", "tokopedia.")
failed_searches = []

class SerperAPIAuthenticationError(RuntimeError): pass
def _domain(url):
    try: return urlparse(url).netloc.lower().replace("www.", "")
    except Exception: return ""
def is_marketplace_url(url): return any(part in _domain(url) for part in MARKETPLACE_DOMAINS)
def canonical_url(url):
    try:
        p = urlparse(url)
        return urlunparse((p.scheme.lower(), p.netloc.lower(), p.path.rstrip('/'), '', '', ''))
    except Exception: return url or ""

def _cache_path(endpoint, payload):
    digest = hashlib.sha256(json.dumps({"endpoint": endpoint, "payload": payload}, sort_keys=True).encode()).hexdigest()
    return os.path.join(CACHE_DIR, f"{digest}.json")

def serper_post(endpoint, payload, use_cache=True):
    cache_path = _cache_path(endpoint, payload)
    if use_cache and os.path.exists(cache_path):
        with open(cache_path, encoding="utf-8") as f: return 200, json.load(f)
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    last_error, last_status = None, None
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(endpoint, headers=headers, json=payload, timeout=REQUEST_TIMEOUT_SEC)
            last_status = response.status_code
            if response.status_code in (401, 403):
                raise SerperAPIAuthenticationError("Serper API key invalid or unauthorized.")
            if response.status_code == 429 or response.status_code in (500, 502, 503):
                last_error = response.text[:500]
                if attempt < MAX_RETRIES - 1: time.sleep(RETRY_BACKOFF_SEC * (2 ** attempt)); continue
            response.raise_for_status()
            data = response.json()
            with open(cache_path, "w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False)
            return response.status_code, data
        except SerperAPIAuthenticationError: raise
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError, ValueError) as exc:
            last_error = str(exc)
            if attempt < MAX_RETRIES - 1: time.sleep(RETRY_BACKOFF_SEC * (2 ** attempt))
    failed_searches.append({"timestamp": datetime.now(timezone.utc).isoformat(), "endpoint": endpoint,
        "query": payload.get("q"), "http_status": last_status, "error": last_error})
    return last_status, None

def search_images_serper(query, num_results=10):
    _, data = serper_post(SERPER_IMAGE_URL, {"q": query, "num": num_results})
    if not data: return []
    results = []
    for item in data.get("images", [])[:num_results]:
        source_url = item.get("link", "")
        results.append({"image_url": item.get("imageUrl") or item.get("thumbnailUrl"), "source_url": source_url,
            "title": item.get("title", ""), "source_domain": item.get("domain") or _domain(source_url)})
    return results

def search_images(query, num=10):
    time.sleep(REQUEST_DELAY_SEC)
    return search_images_serper(query, num)

def search_web_serper(query, num_results=10):
    _, data = serper_post(SERPER_WEB_URL, {"q": query, "num": num_results})
    if not data: return []
    return [{"title": item.get("title", ""), "url": item.get("link", ""), "snippet": item.get("snippet", ""),
             "source_domain": _domain(item.get("link", ""))} for item in data.get("organic", [])[:num_results]]

def search_public_web(query, num=10):
    return [{"listing_url": item["url"], "listing_title": item["title"], "snippet": item["snippet"],
             "source_domain": item["source_domain"]} for item in search_web_serper(query, num)]

print("Serper web/image search functions and response cache ready.")

from bs4 import BeautifulSoup
PAGE_HEADERS={"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36","Accept-Language":"id-ID,id;q=0.9,en-US;q=0.8"}

def _rating(v):
    try:
        m=re.search(r"\d(?:[.,]\d)?",str(v)); return float(m.group(0).replace(",",".")) if m else None
    except Exception: return None

def _walk_json(obj,url,out):
    if isinstance(obj,dict):
        text=next((obj.get(k) for k in ("reviewBody","reviewText","review_text","comment","content") if obj.get(k)),None)
        if text:
            rating=obj.get("ratingValue") or obj.get("rating") or (obj.get("reviewRating") or {}).get("ratingValue")
            out.append({"review_rating":_rating(rating),"review_text":str(text).strip(),"review_date":obj.get("datePublished") or obj.get("date"),"review_url":url})
        for v in obj.values(): _walk_json(v,url,out)
    elif isinstance(obj,list):
        for v in obj: _walk_json(v,url,out)

def extract_reviews_from_page(html,review_url=None):
    """Only extracts review-shaped JSON-LD/__NEXT_DATA__ or explicitly review-named DOM blocks; returns [] when unavailable."""
    soup=BeautifulSoup(html,"html.parser"); found=[]
    for tag in soup.find_all("script"):
        raw=tag.string or tag.get_text(); typ=(tag.get("type") or "").lower()
        if raw and ("ld+json" in typ or tag.get("id")=="__NEXT_DATA__"):
            try: _walk_json(json.loads(raw),review_url,found)
            except Exception: pass
    for block in soup.select('[data-review-id], [class*="review" i], [id*="review" i]'):
        text=block.get_text(" ",strip=True)
        if len(text)>=8: found.append({"review_rating":_rating(text),"review_text":text[:2000],"review_date":None,"review_url":review_url})
    seen=set(); return [r for r in found if not (normalize_review_text(r["review_text"]) in seen or seen.add(normalize_review_text(r["review_text"])))]

def fetch_listing_reviews(url):
    try:
        r=requests.get(url,headers=PAGE_HEADERS,timeout=30)
        if r.status_code in (401,403,429) or r.status_code>=500 or any(x in r.text.lower() for x in ("captcha","access denied","verify you are human")): return "unavailable",[]
        r.raise_for_status(); reviews=extract_reviews_from_page(r.text,url)
        return ("page_review" if reviews else "no_reviews_found"),reviews
    except requests.RequestException: return "unavailable",[]

def record_counterfeit_failure(row,query,source,error,stage):
    failed_searches.append({"timestamp":datetime.now(timezone.utc).isoformat(),"product_id":row.get("product_id"),"query":query,"source":source,"error":error,"stage":stage,"endpoint":None,"http_status":None})


def fetch_evidence_source_images(source_url):
    """Return only images directly declared by the public evidence page; never bypasses access controls."""
    try:
        r=requests.get(source_url,headers=PAGE_HEADERS,timeout=3)
        if r.status_code in (401,403,429) or r.status_code>=500: return []
        r.raise_for_status(); soup=BeautifulSoup(r.text,"html.parser")
        urls=[]
        for prop in ("og:image","twitter:image"):
            for tag in soup.find_all("meta",attrs={"property":prop})+soup.find_all("meta",attrs={"name":prop}):
                if tag.get("content"): urls.append(tag["content"])
        return list(dict.fromkeys(urls))
    except requests.RequestException: return []


Serper web/image search functions and response cache ready.


## 5. Serper API preflight test

Run the next cell before the pipeline. If either request fails, do not proceed.


In [6]:
# Serper API preflight test — run before the main pipeline.
web_query = "COSRX AHA BHA Clarifying Treatment Toner"
image_query = "COSRX AHA BHA Clarifying Treatment Toner official"
try:
    web_status, web_data = serper_post(SERPER_WEB_URL, {"q": web_query, "num": 3}, use_cache=False)
    image_status, image_data = serper_post(SERPER_IMAGE_URL, {"q": image_query, "num": 3}, use_cache=False)
    if not web_data or not image_data: raise RuntimeError("Serper returned no usable response; inspect the status/error above.")
    organic, images = web_data.get("organic", []), image_data.get("images", [])
    print("Web HTTP status:", web_status, "| organic results:", len(organic))
    for item in organic[:3]: print("WEB:", item.get("title"), "|", item.get("link"))
    print("Image HTTP status:", image_status, "| image results:", len(images))
    for item in images[:3]: print("IMAGE:", item.get("imageUrl") or item.get("thumbnailUrl"), "|", item.get("link"))
    SERPER_API_TEST_PASSED = True
    print("✅ Serper API working.")
except Exception as exc:
    SERPER_API_TEST_PASSED = False
    print("Serper API test failed:", type(exc).__name__, str(exc))
    raise


Web HTTP status: 200 | organic results: 3
WEB: AHA/BHA Clarifying Treatment Toner | https://www.cosrx.com/products/aha-bha-clarifying-treatment-toner?srsltid=AfmBOor4EkLjdOuNXHyp0l1PDKR15buw6Jrjf-3PzOt81VDXPL5IwPlt
WEB: Review: COSRX AHA/BHA Clarifying Treatment Toner | https://www.reddit.com/r/AsianBeauty/comments/38k1f9/review_cosrx_ahabha_clarifying_treatment_toner/
WEB: COSRX AHA/BHA Treatment Toner for Whiteheads & ... | https://www.amazon.com/COSRX-Clarifying-Treatment-Toner-150ml/dp/B073P6BPF5
Image HTTP status: 200 | image results: 3
IMAGE: https://www.cosrx.com/cdn/shop/files/ahabha-clarifying-treatment-toner-cosrx-official-1_1024x1024.jpg?v=1724835581 | https://www.cosrx.com/products/aha-bha-clarifying-treatment-toner?srsltid=AfmBOorN5NDRMzgKWwJIXROwXMQWB0GAa-VapMtUCaHIbqXHupBRBZ16
IMAGE: https://media.ulta.com/i/ulta/2504908?w=500&h=500 | https://www.ulta.com/p/ahabha-clarifying-treatment-toner-xlsImpprod15641050?sku=2504908
IMAGE: https://cloudinary.images-iherb.com/image/u

In [7]:
# Compact query strategy: 3 genuine + 5 evidence searches/product (≤728 base searches for 91 products).
GENUINE_QUERY_TEMPLATES = [
    '{brand} {product_name} official', '{brand} {product_name} official website', '{brand} {product_name} Sociolla authorized retailer'
]
EVIDENCE_QUERY_TEMPLATES = [
    '{brand} {product_name} Shopee palsu fake', '{brand} {product_name} Tokopedia palsu fake',
    '{brand} {product_name} Shopee tidak asli tidak original', '{brand} {product_name} Tokopedia tidak asli tidak original',
    '{brand} {product_name} review palsu fake 1 star'
]
def build_queries(brand, product_name, bpom_id, templates):
    return [t.format(brand=brand, product_name=product_name, bpom_id=bpom_id) for t in templates]
print(f"Compact plan: {len(GENUINE_QUERY_TEMPLATES)} genuine + {len(EVIDENCE_QUERY_TEMPLATES)} evidence queries/product.")


Compact plan: 3 genuine + 5 evidence queries/product.


## 6. Download + verification functions

Downloads the actual image bytes (not the webpage), verifies the response is a real
image with Pillow, converts to JPG, and computes hashes for later dedup.

In [8]:
from PIL import Image
import hashlib
import io
import uuid
from datetime import datetime, timezone

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

def download_image(image_url, dest_dir, image_id=None):
    '''
    Downloads image_url, verifies it's a real image, converts to JPG, saves it.
    Returns a dict describing the result (always, even on failure) so callers can
    log it into images.csv / failed_downloads.csv without special-casing.
    '''
    image_id = image_id or uuid.uuid4().hex[:12]
    result = {
        "image_id": image_id,
        "image_url": image_url,
        "image_path": None,
        "download_status": "failed",
        "rejection_reason": None,
        "width": None,
        "height": None,
        "file_size": None,
        "md5_hash": None,
        "downloaded_at": datetime.now(timezone.utc).isoformat(),
    }
    if not image_url:
        result["rejection_reason"] = "empty_url"
        return result

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(image_url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SEC)
            resp.raise_for_status()
            content_type = resp.headers.get("Content-Type", "")
            if "image" not in content_type and not image_url.lower().split("?")[0].endswith(
                (".jpg", ".jpeg", ".png", ".webp")
            ):
                result["rejection_reason"] = f"not_an_image (content-type={content_type})"
                return result

            raw = resp.content
            # Verify it's a genuinely decodable image
            img = Image.open(io.BytesIO(raw))
            img.verify()
            # Re-open after verify() (which invalidates the file handle)
            img = Image.open(io.BytesIO(raw)).convert("RGB")

            width, height = img.size
            os.makedirs(dest_dir, exist_ok=True)
            out_path = os.path.join(dest_dir, f"{image_id}.jpg")
            img.save(out_path, "JPEG", quality=92)

            file_size = os.path.getsize(out_path)
            md5_hash = hashlib.md5(raw).hexdigest()

            result.update({
                "image_path": out_path,
                "download_status": "success",
                "width": width,
                "height": height,
                "file_size": file_size,
                "md5_hash": md5_hash,
            })
            return result
        except Exception as e:
            last_error = str(e)
            time.sleep(RETRY_BACKOFF_SEC * attempt)
            continue

    result["rejection_reason"] = f"download_failed_after_{MAX_RETRIES}_retries: {last_error}"
    return result

print("Download function ready.")

Download function ready.


## 7. Quality filter (resolution, blur, blank/corrupt)

In [9]:
import numpy as np

def compute_blur_score(image_path):
    '''Variance of Laplacian, computed with numpy only (no opencv dependency).
    Lower score = blurrier. Returns None if it can't be computed.'''
    try:
        img = Image.open(image_path).convert("L")
        arr = np.asarray(img, dtype=np.float64)
        # Simple discrete Laplacian kernel via manual convolution
        kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
        h, w = arr.shape
        if h < 3 or w < 3:
            return None
        # Vectorized convolution (valid mode)
        lap = (
            arr[0:-2, 1:-1] + arr[2:, 1:-1] + arr[1:-1, 0:-2] + arr[1:-1, 2:]
            - 4 * arr[1:-1, 1:-1]
        )
        return float(lap.var())
    except Exception:
        return None

BLUR_VARIANCE_MIN = 15.0  # empirical; tune after inspecting a sample of rejects

def quality_check(image_path, width, height):
    '''Returns (passed: bool, reason: str or None).'''
    if width is None or height is None:
        return False, "unreadable_dimensions"
    if width < MIN_WIDTH or height < MIN_HEIGHT:
        return False, f"too_small ({width}x{height})"
    try:
        file_size = os.path.getsize(image_path)
        if file_size < 2000:  # under ~2KB is almost certainly blank/broken
            return False, "file_too_small_likely_blank"
    except OSError:
        return False, "file_missing"

    blur_score = compute_blur_score(image_path)
    if blur_score is not None and blur_score < BLUR_VARIANCE_MIN:
        return False, f"too_blurry (score={blur_score:.1f})"

    return True, None

print("Quality filter ready.")

Quality filter ready.


## 8. Deduplication (MD5 exact + perceptual hash near-duplicate)

In [10]:
import imagehash

def compute_phash(image_path):
    try:
        return str(imagehash.phash(Image.open(image_path)))
    except Exception:
        return None

def dedupe_records(records, phash_threshold=6):
    '''
    records: list of dicts, each must have 'md5_hash' and 'image_path'.
    Mutates nothing; returns (kept_records, duplicate_records) where duplicate_records
    get a 'rejection_reason' set to 'duplicate_of:<image_id>'.
    '''
    seen_md5 = {}
    kept = []
    duplicates = []

    # Pass 1: exact duplicates via MD5
    stage1 = []
    for rec in records:
        md5 = rec.get("md5_hash")
        if md5 and md5 in seen_md5:
            rec["rejection_reason"] = f"duplicate_of:{seen_md5[md5]}"
            duplicates.append(rec)
        else:
            if md5:
                seen_md5[md5] = rec["image_id"]
            stage1.append(rec)

    # Pass 2: near-duplicates via perceptual hash (within same product+label group,
    # since two different-but-visually-similar products shouldn't be merged)
    groups = {}
    for rec in stage1:
        key = (rec.get("product_id"), rec.get("label"))
        groups.setdefault(key, []).append(rec)

    for key, group in groups.items():
        kept_hashes = []  # list of (image_id, imagehash.ImageHash)
        for rec in group:
            phash_str = compute_phash(rec["image_path"]) if rec.get("image_path") else None
            rec["phash"] = phash_str
            if not phash_str:
                kept.append(rec)
                continue
            h = imagehash.hex_to_hash(phash_str)
            is_dupe = False
            for kept_id, kept_hash in kept_hashes:
                if h - kept_hash <= phash_threshold:
                    rec["rejection_reason"] = f"near_duplicate_of:{kept_id}"
                    duplicates.append(rec)
                    is_dupe = True
                    break
            if not is_dupe:
                kept_hashes.append((rec["image_id"], h))
                kept.append(rec)

    return kept, duplicates

print("Dedup functions ready.")

Dedup functions ready.


## 9. OCR + brand/product/BPOM matching

In [11]:
import pytesseract
from rapidfuzz import fuzz

def run_ocr(image_path):
    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return ""

BPOM_PATTERN = re.compile(r"\b([A-Z]{2}\d{9,12})\b")

def extract_ocr_fields(ocr_text, master_brand, master_product_name, master_size):
    ocr_upper = ocr_text.upper()

    bpom_matches = BPOM_PATTERN.findall(ocr_upper)
    ocr_bpom_id = bpom_matches[0] if bpom_matches else None

    brand_score = fuzz.partial_ratio(str(master_brand).upper(), ocr_upper)
    product_score = fuzz.partial_ratio(str(master_product_name).upper(), ocr_upper)

    size_match = None
    if master_size:
        size_str = str(master_size).upper().replace(" ", "")
        size_match = size_str in ocr_upper.replace(" ", "")

    return {
        "ocr_text": ocr_text[:2000],  # cap length for CSV sanity
        "ocr_brand_score": brand_score,
        "ocr_product_name_score": product_score,
        "ocr_size_match": size_match,
        "ocr_bpom_id": ocr_bpom_id,
    }

def bpom_match_status(master_bpom_id, ocr_bpom_id):
    if not ocr_bpom_id:
        return "not_visible"
    master_clean = re.sub(r"\s+", "", str(master_bpom_id).upper())
    ocr_clean = re.sub(r"\s+", "", str(ocr_bpom_id).upper())
    if master_clean == ocr_clean:
        return "match"
    if master_clean and master_clean in ocr_clean:
        return "match"
    return "mismatch"

print("OCR functions ready.")

OCR functions ready.


## 10. Counterfeit-review evidence classification

Evidence must contain counterfeit-related text. A low rating by itself is never enough. `counterfeit_confirmed` is not assigned anywhere in this notebook.


In [12]:
COUNTERFEIT_PHRASES=["barang palsu","produk palsu","tidak asli","tidak original","bukan original","bukan ori","tidak ori","kemasan berbeda","kemasannya beda","beda dengan asli","beda dengan official","beda dengan official store","isi berbeda","tekstur berbeda","warna berbeda","bau berbeda","bpom tidak sesuai","bpom tidak cocok","diduga palsu","terindikasi palsu","counterfeit","not authentic","not genuine","not original","replica","packaging different","different from official","fake product","counterfeit product","palsu","fake","kw"]
EXPLICIT_TERMS=["palsu","fake","counterfeit","replica"]
AUTH_TERMS=["tidak asli","tidak original","bukan original","bukan ori","tidak ori","not authentic","not genuine","not original"]
MISMATCH_TERMS=["kemasan berbeda","kemasannya beda","beda dengan asli","beda dengan official","beda dengan official store","packaging different","different from official"]
def normalize_review_text(text):
 text=unicodedata.normalize("NFKD",str(text or "")).lower(); text="".join(c for c in text if not unicodedata.combining(c)); text=re.sub(r"[^a-z0-9]+"," ",text); return re.sub(r"\s+"," ",text).strip()
def find_evidence_keywords(text):
 n=normalize_review_text(text); return [x for x in COUNTERFEIT_PHRASES if normalize_review_text(x) in n]
def classify_review_evidence(text,review_rating=None,independent_reports=1,bpom_mismatch=False,visual_mismatch=False):
 n=normalize_review_text(text); keys=find_evidence_keywords(text)
 if not keys:return False,"NO_EVIDENCE",0,[]
 score=3*any(normalize_review_text(x) in n for x in EXPLICIT_TERMS)+2*any(normalize_review_text(x) in n for x in AUTH_TERMS)+2*any(normalize_review_text(x) in n for x in MISMATCH_TERMS)+int(review_rating is not None and float(review_rating)<=2)+int(independent_reports>1)+int(bpom_mismatch)+int(visual_mismatch)
 return True,("STRONG_EVIDENCE" if independent_reports>1 or score>=5 else "MODERATE_EVIDENCE" if score>=3 else "WEAK_EVIDENCE"),score,keys
def assign_label_and_reason(search_type,ocr_fields,bpom_status):
 if bpom_status=="mismatch":return "unknown",True,"BPOM mismatch between master data and OCR"
 return ("genuine_reference",False,None) if search_type=="genuine" else ("reported_counterfeit_candidate",False,None)
print("Evidence classifier ready: text is primary; rating is supporting only.")


PUBLIC_EVIDENCE_QUERY_TEMPLATES=[
 '{brand} {product_name} palsu','{brand} {product_name} fake','{brand} {product_name} counterfeit',
 '{brand} {product_name} tidak asli','{brand} {product_name} tidak original','{brand} {product_name} barang palsu',
 '{brand} {product_name} produk palsu','{brand} {product_name} kemasan berbeda','{brand} {product_name} review palsu',
 '{brand} {product_name} fake product review'
]
def classify_source_type(url,title,snippet):
 d=_domain(url); text=(title+' '+snippet).lower()
 if is_marketplace_url(url): return 'marketplace'
 if 'reddit.com' in d: return 'reddit'
 if any(x in d for x in ('forum','kaskus','female-daily','femaledaily')): return 'forum'
 if any(x in d for x in ('news','kompas','detik','tempo','cnn','tribun')): return 'news'
 if any(x in d for x in ('instagram','facebook','tiktok','x.com','twitter.com')): return 'social'
 if 'review' in text or 'ulasan' in text: return 'review'
 if any(x in d for x in ('blog','wordpress','medium.com')): return 'blog'
 return 'unknown'
print(f"Public evidence discovery ready: {len(PUBLIC_EVIDENCE_QUERY_TEMPLATES)} evidence queries/product.")


Evidence classifier ready: text is primary; rating is supporting only.
Public evidence discovery ready: 10 evidence queries/product.


## 11. Main pipeline: genuine search + evidence-gated marketplace collection

The marketplace branch first finds explicit counterfeit-related public review/listing evidence. It only downloads images whose search result points to that exact listing URL; listings without usable evidence are recorded but never used as counterfeit samples.


In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed
if not SERPER_API_TEST_PASSED:
    raise RuntimeError("Run Cell 11 and resolve the Serper API test before running the pipeline.")

from tqdm.auto import tqdm

all_image_records, failed_downloads, products_shortage, manual_review_records = [], [], [], []
review_evidence_records, marketplace_listing_records = [], []
public_evidence_search_records, reported_candidate_image_records = [], []

def base_record(row):
    return {k: row.get(k) for k in ["product_id", "brand", "product_name", "bpom_id"]}

def process_image_hit(row, hit, dest_dir, source_type, extra=None):
    dl = download_image(hit.get("image_url"), dest_dir)
    dl.update(base_record(row)); dl.update({"source_url": hit.get("source_url"), "source_domain": hit.get("source_domain"),
        "image_url": hit.get("image_url"), "source_type": source_type, "marketplace": _domain(hit.get("source_url", ""))})
    if extra: dl.update(extra)
    if dl["download_status"] != "success":
        dl["label"] = "rejected"; return None, dl
    passed, reason = quality_check(dl["image_path"], dl["width"], dl["height"])
    if not passed:
        dl.update({"download_status": "rejected", "rejection_reason": reason, "label": "rejected"})
        try: os.remove(dl["image_path"])
        except OSError: pass
        return None, dl
    fields = extract_ocr_fields(run_ocr(dl["image_path"]), row["brand"], row["product_name"], row.get("size"))
    status = bpom_match_status(row["bpom_id"], fields["ocr_bpom_id"])
    label, needs_review, reason = assign_label_and_reason(source_type, fields, status)
    dl.update({**fields, "bpom_match_status": status, "label": label})
    if needs_review: manual_review_records.append({"image_path": dl["image_path"], "product_id": row["product_id"], "reason": reason})
    return dl, None

def collect_genuine(row):
    records, failures, seen = [], [], set()
    dest = os.path.join(IMAGES_DIR, "genuine_reference", row["product_id"])
    for query in build_queries(row["brand"], row["product_name"], row["bpom_id"], GENUINE_QUERY_TEMPLATES):
        if len(records) >= GENUINE_TARGET: break
        for hit in search_images(query, RESULTS_PER_QUERY):
            if len(records) >= GENUINE_TARGET or hit.get("image_url") in seen: continue
            seen.add(hit.get("image_url")); rec, failure = process_image_hit(row, hit, dest, "genuine", {"search_query": query})
            if rec: records.append(rec)
            if failure: failures.append(failure)
    return records, failures

def discover_marketplace_listings(row):
 listings=[]; seen=set()
 for market in ("Shopee","Tokopedia"):
  query=f"{row['brand']} {row['product_name']} {market}"; results=search_public_web(query,RESULTS_PER_QUERY)
  if not results: record_counterfeit_failure(row,query,market,"no_listing_results","marketplace_discovery")
  for r in results:
   url=r["listing_url"]
   if is_marketplace_url(url) and canonical_url(url) not in seen:
    seen.add(canonical_url(url)); listings.append({**base_record(row),"marketplace":_domain(url),"listing_url":url,"listing_title":r["listing_title"],"seller_name":None,"listing_status":"marketplace_listing_candidate","review_access":None,"review_count":0,"counterfeit_evidence_found":False})
 return listings

def collect_counterfeit_evidence(row):
    """Public evidence is independent of marketplace-page accessibility; image creation remains source-linked."""
    records=[]; failures=[]; evidence_rows=[]; search_rows=[]
    listings=discover_marketplace_listings(row)  # discovery is metadata only, never a counterfeit label
    queries=build_queries(row["brand"],row["product_name"],row["bpom_id"],PUBLIC_EVIDENCE_QUERY_TEMPLATES)
    seen=set(); stats={"public":0,"matched":0,"sources":set(),"linked":0}
    for query in queries:
        results=search_public_web(query,RESULTS_PER_QUERY)
        if not results: record_counterfeit_failure(row,query,"serper","no_public_evidence_results","public_evidence_search")
        for result in results:
            source_url=result["listing_url"]; title=result["listing_title"]; snippet=result["snippet"]
            source_type=classify_source_type(source_url,title,snippet); combined=(title+" "+snippet).strip()
            ok,level,score,keys=classify_review_evidence(combined,None)
            rejection_reason=None if ok else "no_counterfeit_related_text"
            search_row={**base_record(row),"query":query,"source_url":source_url,"source_domain":_domain(source_url),"title":title,"snippet":snippet,"source_type":source_type,"evidence_keyword":"; ".join(keys),"evidence_level":level,"evidence_score":score,"accepted_as_evidence":ok,"rejection_reason":rejection_reason}
            search_rows.append(search_row); stats["public"]+=1
            if not ok: continue
            fingerprint=(canonical_url(source_url),normalize_review_text(combined))
            if fingerprint in seen or stats["matched"] >= MAX_PUBLIC_EVIDENCE_PER_PRODUCT: continue
            seen.add(fingerprint); stats["matched"]+=1; stats["sources"].add(source_url)
            access="indexed_public_evidence"
            evidence={**base_record(row),"source_type":source_type,"source_domain":_domain(source_url),"source_url":source_url,"title":title,"snippet":snippet,
                "marketplace":_domain(source_url) if is_marketplace_url(source_url) else None,"listing_url":source_url if is_marketplace_url(source_url) else None,"listing_title":title if is_marketplace_url(source_url) else None,"seller_name":None,
                "review_url":None,"review_rating":None,"review_text":combined,"review_date":None,"review_access":access,"evidence_keyword":"; ".join(keys),"evidence_level":level,"evidence_score":score,"linked_image_url":None,"image_url":None,"image_path":None,"label":None,"evidence_status":"evidence_without_image"}
            # Priority 1/3: image explicitly declared by the evidence source page.
            hits=[{"image_url":u,"source_url":source_url,"source_domain":_domain(source_url),"title":title} for u in fetch_evidence_source_images(source_url)]
            # Priority 2: source-URL exact image search. This is also the only permitted marketplace fallback.
            if not hits:
                hits=[h for h in search_images(source_url,RESULTS_PER_QUERY) if canonical_url(h.get("source_url",""))==canonical_url(source_url)]
            if not hits:
                record_counterfeit_failure(row,source_url,source_type,"no_source_linked_image","evidence_without_image");evidence_rows.append(evidence);continue
            for hit in hits:
                if len(records)>=COUNTERFEIT_MAX_CANDIDATES: break
                extra={"search_query":source_url,**{k:evidence.get(k) for k in ("listing_url","listing_title","seller_name","review_url","review_rating","review_text","review_date","evidence_keyword","evidence_level","evidence_score")}}
                rec,failure=process_image_hit(row,hit,os.path.join(IMAGES_DIR,"reported_counterfeit_candidate",row["product_id"]),"counterfeit_evidence",extra)
                if rec:
                    records.append(rec);stats["linked"]+=1;evidence.update({"linked_image_url":rec["image_url"],"image_url":rec["image_url"],"image_path":rec["image_path"],"label":rec["label"],"evidence_status":"image_linked"});break
                if failure: failures.append(failure);evidence["evidence_status"]="image_download_failed"
            evidence_rows.append(evidence)
    print(f"\nPRODUCT: {row['brand']} {row['product_name']}\nMarketplace listings found: {len(listings)}\nPublic evidence search results: {stats['public']}\nEvidence-related results: {stats['matched']}\nEvidence sources: {len(stats['sources'])}\nCounterfeit evidence records: {len(evidence_rows)}\nImages linked to evidence: {stats['linked']}\nReported counterfeit candidate images: {len(records)}")
    if evidence_rows:
        for e in evidence_rows: print("EVIDENCE |",e["source_type"],e["source_domain"],e["source_url"],e["title"],e["evidence_keyword"],e["evidence_score"],e["evidence_level"])
    else:
        for r in search_rows[:5]: print("REJECTED RESULT |",r["source_domain"],r["title"],"|",r["snippet"],"|",r["rejection_reason"])
    return records,failures,evidence_rows,listings,search_rows

def process_one_product(row):
    genuine, genuine_failed = collect_genuine(row)
    candidate, candidate_failed, evidence, listings, search_rows = collect_counterfeit_evidence(row)
    if len(genuine) < 5: products_shortage.append({**base_record(row), "search_type": "genuine", "collected": len(genuine), "target": GENUINE_TARGET})
    return genuine + candidate, genuine_failed + candidate_failed, evidence, listings, search_rows

rows = master_df.head(PILOT_PRODUCTS).to_dict("records") if PILOT_MODE else master_df.to_dict("records")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_one_product, row): row["product_id"] for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Products"):
        pid = futures[future]
        try:
            recs, failed, evidence, listings, search_rows = future.result()
            all_image_records.extend(recs); failed_downloads.extend(failed); review_evidence_records.extend(evidence); marketplace_listing_records.extend(listings); public_evidence_search_records.extend(search_rows); reported_candidate_image_records.extend(recs)
        except Exception as e: print(f"Product {pid} failed without stopping the pipeline: {e}")
print(f"Processed {len(rows)} products; examined {len(marketplace_listing_records)} marketplace listings.")


Products:   0%|          | 0/91 [00:00<?, ?it/s]


PRODUCT: Dear Klairs Freshly Juiced Vitamin Skin Prep Pads
Marketplace listings found: 3
Public evidence search results: 87
Evidence-related results: 0
Evidence sources: 0
Counterfeit evidence records: 0
Images linked to evidence: 0
Reported counterfeit candidate images: 0
REJECTED RESULT | skincupid.us KLAIRS Freshly Juiced Vitamin Skin Prep Pads (80 Pads) | Treats the various problems behind poor makeup adhesion, including dehydration, enlarged pores, rough texture, and irritation; Fades dark spots, brightens skin ... | no_counterfeit_related_text
REJECTED RESULT | koreanskincare.com Dear Klairs - Freshly Juiced Vitamin Skin Prep Pads | The Dear Klairs Freshly Juiced Vitamin Skin Prep Pads are multifunctional skincare pads designed to refresh, smooth, and brighten the complexion while ... | no_counterfeit_related_text
REJECTED RESULT | reddit.com Dear Klairs - Freshly Juiced Vitamin toner pads | Yes, they can be used daily, but for dry and sensitive skin. They're hydrating with a mi

c:\Users\Richelle Marvela\anaconda3_2\envs\gpuenv\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



PRODUCT: Hayejin Blessing Of Sprout Radiance Toner
Marketplace listings found: 6
Public evidence search results: 90
Evidence-related results: 0
Evidence sources: 0
Counterfeit evidence records: 0
Images linked to evidence: 0
Reported counterfeit candidate images: 0
REJECTED RESULT | stellangelita.blogspot.com Hayejin Blessing of Sprouts Radiance Toner | Hayejin Blessing of Sprout Radiance Toner is a toner that can be used to hydrate and brighten your skin at the same time. This toner is ... | no_counterfeit_related_text
REJECTED RESULT | youtube.com WHY DOES EVERYONE LOVE HAYEJIN? | me on ig @tasyafarasya. SHOP NAME: Hayejin Korea ... WHY DOES EVERYONE LOVE HAYEJIN? SKINCARE I USE DILIGENTLY BUT RARELY DISCUSS! | no_counterfeit_related_text
REJECTED RESULT | sparkleandnargles.wordpress.com Review | Hayejin Blessing of Sprout Radiance Toner | ✔️True to its name, it really makes the skin radiant and bright. Even in the first few days of using it, you would already notice how it makes ..

## 12. Deduplicate

In [14]:
kept_records, duplicate_records = dedupe_records(all_image_records)

for dup in duplicate_records:
    try:
        if dup.get("image_path") and os.path.exists(dup["image_path"]):
            os.remove(dup["image_path"])
    except OSError:
        pass

print(f"Kept {len(kept_records)} unique images, removed {len(duplicate_records)} duplicates.")

Kept 706 unique images, removed 40 duplicates.


## 13. Write metadata CSVs

In [15]:
IMAGES_CSV_COLUMNS = [
    "image_id", "product_id", "brand", "product_name", "bpom_id", "image_url", "image_path", "source_url", "source_domain", "marketplace",
    "listing_url", "listing_title", "seller_name", "review_url", "review_rating", "review_text", "review_date",
    "evidence_keyword", "evidence_level", "evidence_score", "label", "ocr_text", "ocr_brand_score", "ocr_product_name_score",
    "ocr_size_match", "ocr_bpom_id", "bpom_match_status", "md5_hash", "phash", "download_status", "rejection_reason",
    "source_type", "search_query", "width", "height", "file_size", "downloaded_at"
]
REVIEW_COLUMNS = ["product_id", "brand", "product_name", "bpom_id", "marketplace", "listing_url", "listing_title", "seller_name",
                  "review_url", "review_rating", "review_text", "review_date", "review_access", "evidence_keyword", "evidence_level", "evidence_score", "linked_image_url"]

def frame_with_columns(records, columns):
    df = pd.DataFrame(records)
    for col in columns:
        if col not in df.columns: df[col] = None
    return df[columns]

images_df = frame_with_columns(kept_records, IMAGES_CSV_COLUMNS)
rejected_df = frame_with_columns(duplicate_records + failed_downloads, IMAGES_CSV_COLUMNS)
review_evidence_df = frame_with_columns(review_evidence_records, REVIEW_COLUMNS)
images_df.to_csv(os.path.join(META_DIR, "images.csv"), index=False)
rejected_df.to_csv(os.path.join(META_DIR, "rejected_images.csv"), index=False)
pd.DataFrame(failed_downloads).to_csv(os.path.join(META_DIR, "failed_downloads.csv"), index=False)
pd.DataFrame(failed_searches, columns=["timestamp", "product_id", "query", "source", "error", "stage", "endpoint", "http_status"]).to_csv(os.path.join(META_DIR, "failed_searches.csv"), index=False)
pd.DataFrame(products_shortage).to_csv(os.path.join(META_DIR, "products_without_enough_images.csv"), index=False)
pd.DataFrame(manual_review_records).to_csv(os.path.join(META_DIR, "manual_review.csv"), index=False)
review_evidence_df.to_csv(os.path.join(META_DIR, "review_evidence.csv"), index=False)
EVIDENCE_COLUMNS=["product_id","brand","product_name","bpom_id","source_type","source_domain","source_url","title","snippet","evidence_keyword","evidence_level","evidence_score","review_rating","review_access","image_url","image_path","label","evidence_status"]
counterfeit_evidence_df=frame_with_columns(review_evidence_records,EVIDENCE_COLUMNS)
counterfeit_evidence_df.to_csv(os.path.join(META_DIR, "counterfeit_evidence.csv"),index=False)
pd.DataFrame(public_evidence_search_records).to_csv(os.path.join(META_DIR,"evidence_search_results.csv"),index=False)
frame_with_columns(reported_candidate_image_records,IMAGES_CSV_COLUMNS).to_csv(os.path.join(META_DIR,"reported_counterfeit_candidate_images.csv"),index=False)
pd.DataFrame(marketplace_listing_records).to_csv(os.path.join(META_DIR,"marketplace_listings.csv"),index=False)
images_df[["image_id", "product_id", "source_url", "source_domain", "listing_url", "review_url"]].to_csv(os.path.join(META_DIR, "sources.csv"), index=False)
master_df.to_csv(os.path.join(META_DIR, "products.csv"), index=False)
print("Wrote evidence-aware metadata CSVs, including review_evidence.csv and counterfeit_evidence.csv.")


Wrote evidence-aware metadata CSVs, including review_evidence.csv and counterfeit_evidence.csv.


## 14. Final summary

In [16]:
total_products = len(rows)
genuine_count = int((images_df.label == "genuine_reference").sum())
candidate_count = int((images_df.label == "reported_counterfeit_candidate").sum())
unknown_count = int((images_df.label == "unknown").sum())
evidence_count = len(review_evidence_df)
summary = {
    "products_processed": total_products, "total_products_in_master_dataset": n_products,
    "genuine_images": genuine_count, "marketplace_listings_examined": len(marketplace_listing_records),
    "listings_with_evidence": evidence_count, "reviews_evidence_snippets_found": evidence_count,
    "counterfeit_related_evidence": evidence_count, "reported_counterfeit_candidate_images": candidate_count,
    "unknown": unknown_count, "rejected": len(rejected_df), "failed_downloads": len(failed_downloads),
}
print("SAFECART DATASET SUMMARY")
for key, value in summary.items(): print(f"{key.replace('_', ' ').upper():42}: {value}")
if evidence_count:
    print("\nStrongest evidence examples:")
    display(review_evidence_df.sort_values("evidence_score", ascending=False)[["product_name", "marketplace", "listing_url", "review_rating", "review_text", "evidence_keyword", "evidence_level", "linked_image_url"]].head(10))
per_product = images_df.groupby(["product_id", "brand", "product_name", "bpom_id", "label"]).size().unstack(fill_value=0).reset_index()
for col in ["genuine_reference", "reported_counterfeit_candidate", "unknown"]:
    if col not in per_product: per_product[col] = 0
per_product.to_csv(os.path.join(META_DIR, "dataset_summary.csv"), index=False)


SAFECART DATASET SUMMARY
PRODUCTS PROCESSED                        : 91
TOTAL PRODUCTS IN MASTER DATASET          : 91
GENUINE IMAGES                            : 686
MARKETPLACE LISTINGS EXAMINED             : 634
LISTINGS WITH EVIDENCE                    : 82
REVIEWS EVIDENCE SNIPPETS FOUND           : 82
COUNTERFEIT RELATED EVIDENCE              : 82
REPORTED COUNTERFEIT CANDIDATE IMAGES     : 20
UNKNOWN                                   : 0
REJECTED                                  : 457
FAILED DOWNLOADS                          : 417

Strongest evidence examples:


,product_name,marketplace,listing_url,review_rating,review_text,evidence_keyword,evidence_level,linked_image_url
0,AHA/BHA Clarifying treatment toner,None,None,None,Cosrx Toner Real Vs Fake Perbedaan COSRX Asli ...,palsu; fake,MODERATE_EVIDENCE,None
61,Daily Boost Antioxidant Serum,None,None,None,Perbedaan Elsheskin Radiant Serum Asli Dan Pal...,palsu,MODERATE_EVIDENCE,None
59,Dr.G RTX Into Serum Vitaminshot,None,None,None,RTX Into Serum VitaminShot 2-Week Program by D...,counterfeit,MODERATE_EVIDENCE,https://cdn-image.oliveyoung.com/prdtImg/1765/...
58,The Hyaluronic Acid 3 Serum,None,None,None,Real vs Fake: Cosrx Snail 96 Mucin Essence | S...,fake,MODERATE_EVIDENCE,None
57,Retinol Rejuvenating Serum,None,None,None,Waspada Produk Lanbena Palsu Untuk Skincare Ke...,palsu,MODERATE_EVIDENCE,https://p19-common-sign.tiktokcdn-us.com/tos-a...
56,15% Vitamin C + Glow Advance Serum,None,None,None,Perbedaan Elsheskin Radiant Serum Asli Dan Pal...,palsu,MODERATE_EVIDENCE,None
55,Concentrated Ginseng Brightening Serum,None,None,None,How to Tell If Sulwhasoo Is Fake: 5 Checks | S...,fake,MODERATE_EVIDENCE,https://tjzhhfczyjvfjjmuvegd.supabase.co/stora...
54,Vinopure Blemish Salicylic Serum,None,None,None,How to Apply Vinopure Blemish Control Salicyli...,fake,MODERATE_EVIDENCE,https://p16-common-sign.tiktokcdn-us.com/tos-u...
53,Perfect Whip Collagen in,None,None,None,4-step pink skincare 🎀 @Senka by FineToday Per...,palsu,MODERATE_EVIDENCE,None
52,PDRN Hyaluronic Acid Capsule 100 Serum,None,None,None,📢 Penting untuk kamu yang beli Anua PDRN Serum...,produk palsu; palsu,MODERATE_EVIDENCE,None


## 15. README + ZIP the dataset

In [17]:
readme_text = f"""# SafeCart Dataset

Generated on {datetime.now(timezone.utc).isoformat()} from {os.path.basename(MASTER_XLSX_PATH)}.

Search provider: Serper API. Search responses are cached in `cache/`; API credentials are never written to outputs.
Marketplace images enter the dataset only when a counterfeit-related public snippet is traceable to the same marketplace listing. Normal marketplace listings and low ratings alone are never counterfeit samples.

Labels: `genuine_reference`, `marketplace_no_counterfeit_evidence` (metadata-only), `reported_counterfeit_candidate`, `unknown`, and `rejected`. `counterfeit_confirmed` is manual-only.
"""
with open(os.path.join(BASE_DIR, "README.md"), "w", encoding="utf-8") as f: f.write(readme_text)
import shutil
if os.path.exists("SafeCart_Dataset.zip"): os.remove("SafeCart_Dataset.zip")
shutil.make_archive("SafeCart_Dataset", "zip", BASE_DIR)
print("Created SafeCart_Dataset.zip")


Created SafeCart_Dataset.zip


In [18]:
# Optional deterministic validation of the required review distinction.
example_fake = "Barangnya palsu, kemasan berbeda dengan yang saya beli di official store."
example_good = "Barang bagus dan original."
example_service = "Pengiriman lama dan seller tidak responsif."
for text in [example_fake, example_good, example_service]:
    print(text, "->", classify_review_evidence(text, review_rating=1)[:3])
assert classify_review_evidence(example_fake, 1)[0] is True
assert classify_review_evidence(example_good, 5)[0] is False
assert classify_review_evidence(example_service, 1)[0] is False
print("Validation passed.")


Barangnya palsu, kemasan berbeda dengan yang saya beli di official store. -> (True, 'STRONG_EVIDENCE', 6)
Barang bagus dan original. -> (False, 'NO_EVIDENCE', 0)
Pengiriman lama dan seller tidak responsif. -> (False, 'NO_EVIDENCE', 0)
Validation passed.


In [ ]:
# Serper API preflight test is Cell 11; run it before the pipeline.
